1: MNIST Data Preparation and Client Partitioning




In [ ]:
# ============================================================
# Purpose:
# - Create deterministic MNIST client splits for federated learning
# - Simulate heterogeneous clients by removing 3 to 5 classes per client
# - Build train/validation datasets for each client
# - Plot per-client label distributions for inspection
#
# Outputs:
#   client_train_datasets
#   client_val_datasets
#   test_ds
#   client_idxs
#   client_present
#   y_train
# ============================================================

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ---------------- Configuration ----------------
NUM_CLIENTS = 10
NUM_CLASSES = 10
BATCH_SIZE = 64
VAL_FRAC = 0.1
SEED = 42

MISSING_MIN = 3
MISSING_MAX = 5
ALPHA_WITHIN = 0.2

# Set random seeds for reproducibility
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- Client Partitioning ----------------
def make_clients_missing_classes(y, num_clients, num_classes,
                                 missing_min, missing_max,
                                 alpha_within, seed):
    """
    Create deterministic client splits with missing classes.

    Each client is assigned a subset of classes, with the number of
    missing classes sampled between missing_min and missing_max.
    For each class, samples are distributed only among clients that
    contain that class, using a Dirichlet allocation.

    Parameters
    ----------
    y : np.ndarray
        Training labels.
    num_clients : int
        Number of clients.
    num_classes : int
        Total number of classes.
    missing_min : int
        Minimum number of classes missing per client.
    missing_max : int
        Maximum number of classes missing per client.
    alpha_within : float
        Dirichlet concentration parameter controlling within-class skew.
    seed : int
        Random seed for deterministic splits.

    Returns
    -------
    client_idxs : list of lists
        Sample indices assigned to each client.
    client_present : list of lists
        Classes available on each client.
    """
    rng = np.random.default_rng(seed)
    y = y.reshape(-1)

    # Select the set of classes available on each client
    client_present = []
    for _ in range(num_clients):
        m = int(rng.integers(missing_min, missing_max + 1))
        present = rng.choice(num_classes, size=num_classes - m, replace=False)
        client_present.append(sorted(present.tolist()))

    # Build a lookup from class to eligible clients
    class_to_clients = {c: [] for c in range(num_classes)}
    for k, cls_list in enumerate(client_present):
        for c in cls_list:
            class_to_clients[c].append(k)

    # Distribute each class only to clients that contain it
    client_idxs = [[] for _ in range(num_clients)]
    for c in range(num_classes):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)

        eligible_clients = class_to_clients[c]
        if len(eligible_clients) == 0:
            raise RuntimeError(f"No clients contain class {c}. Please adjust the parameters or seed.")

        proportions = rng.dirichlet(alpha_within * np.ones(len(eligible_clients)))
        cuts = (np.cumsum(proportions) * len(idx)).astype(int)
        splits = np.split(idx, cuts[:-1])

        for j, k in enumerate(eligible_clients):
            client_idxs[k].extend(splits[j].tolist())

    # Shuffle each client's final index list
    for k in range(num_clients):
        rng.shuffle(client_idxs[k])

    return client_idxs, client_present


def make_tf_dataset(X, y, idx, batch_size=64, shuffle=True):
    """
    Build a TensorFlow dataset from a subset of samples.
    """
    ds = tf.data.Dataset.from_tensor_slices((X[idx], y[idx]))
    if shuffle:
        ds = ds.shuffle(min(len(idx), 20000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def build_mnist_clients():
    """
    Load MNIST, apply preprocessing, split the data across clients,
    and create train/validation/test datasets.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()
    y_tr = y_tr.reshape(-1).astype("int32")
    y_te = y_te.reshape(-1).astype("int32")

    # Scale pixel values to [0, 1] and add a channel dimension
    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    # Normalize using training-set statistics
    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7
    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    # Create deterministic heterogeneous client partitions
    client_idxs, client_present = make_clients_missing_classes(
        y_tr,
        num_clients=NUM_CLIENTS,
        num_classes=NUM_CLASSES,
        missing_min=MISSING_MIN,
        missing_max=MISSING_MAX,
        alpha_within=ALPHA_WITHIN,
        seed=SEED
    )

    client_train_datasets, client_val_datasets = [], []

    for k in range(NUM_CLIENTS):
        idx = np.array(client_idxs[k])
        n = len(idx)

        if n < 10:
            raise RuntimeError(f"Client {k} has too few samples ({n}). Try a different alpha or seed.")

        n_val = max(1, int(VAL_FRAC * n))
        val_idx = idx[:n_val]
        tr_idx = idx[n_val:]

        client_train_datasets.append(
            make_tf_dataset(x_tr, y_tr, tr_idx, batch_size=BATCH_SIZE, shuffle=True)
        )
        client_val_datasets.append(
            make_tf_dataset(x_tr, y_tr, val_idx, batch_size=BATCH_SIZE, shuffle=False)
        )

    test_ds = (
        tf.data.Dataset.from_tensor_slices((x_te, y_te))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    return client_train_datasets, client_val_datasets, test_ds, client_idxs, client_present, y_tr


client_train_datasets, client_val_datasets, test_ds, client_idxs, client_present, y_train = build_mnist_clients()

print("MNIST client datasets created successfully with fixed seed =", SEED)
for k in range(NUM_CLIENTS):
    missing = sorted(list(set(range(NUM_CLASSES)) - set(client_present[k])))
    print(f"Client {k:02d}: present={client_present[k]} | missing={missing} | n={len(client_idxs[k])}")

# ---------------- Label Distribution Plots ----------------
def plot_client_distribution(y, idxs, title):
    """
    Plot the class distribution for each client.
    """
    cols = 5
    rows = int(np.ceil(len(idxs) / cols))

    plt.figure(figsize=(18, 10))
    for k, idx in enumerate(idxs):
        labels = y[np.array(idx)]
        unique_labels, counts = np.unique(labels, return_counts=True)

        ax = plt.subplot(rows, cols, k + 1)
        ax.bar(unique_labels, counts)
        ax.set_title(f"Client {k}", fontsize=10)
        ax.set_xticks(range(NUM_CLASSES))

    plt.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


plot_client_distribution(
    y_train,
    client_idxs,
    title=f"MNIST per-client label distribution (each client missing {MISSING_MIN} to {MISSING_MAX} classes)"
)

2: Visualizing the Non-IID Client Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_noniid_heatmap(y, client_idxs, num_clients, num_classes, title):
    """
    Plot a heatmap showing the class distribution across clients.

    Each row corresponds to one client, and each column corresponds
    to one class label. The cell values represent the number of
    samples of each class assigned to each client.
    """
    counts = np.zeros((num_clients, num_classes), dtype=np.int64)

    for client_id in range(num_clients):
        labels = y[np.array(client_idxs[client_id])]
        unique_labels, class_counts = np.unique(labels, return_counts=True)
        counts[client_id, unique_labels] = class_counts

    fig, ax = plt.subplots(figsize=(4, 6))
    im = ax.imshow(counts, aspect="auto")

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Class label", fontweight="bold")
    ax.set_ylabel("Client ID", fontweight="bold")

    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_clients))

    ax.set_xticklabels(range(num_classes), fontweight="bold")
    ax.set_yticklabels(range(num_clients), fontweight="bold")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Number of samples", fontweight="bold")
    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")

    plt.tight_layout()
    plt.show()

    return counts


# Generate the class-distribution heatmap
counts = plot_noniid_heatmap(
    y_train,
    client_idxs,
    NUM_CLIENTS,
    NUM_CLASSES,
    f"MNIST label distribution across clients (missing {MISSING_MIN} to {MISSING_MAX} classes)"
)

3: FedAvg Training on MNIST

In [ ]:
# ============================================================
# Purpose:
# - Train a global FedAvg model using the client datasets from Cell 1
# - Track round-level training and validation performance using local metrics
# - Apply early stopping based on mean training accuracy across clients
# - Evaluate on the test set only after training is complete
# - Save the trained global model and metric history
#
# Saves:
#   MNIST_FedAvg_02.h5
#   FedAvg_metrics_rows_02.csv
# ============================================================

import os
import csv
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Training Configuration ----------------
ROUNDS = 20
BATCHES_PER_ROUND = None
LR_BACKBONE = 1e-3

# Early stopping is based only on round-level training accuracy
EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0   # Increase slightly if small fluctuations should be ignored

SAVE_DIR = "/content/drive/MyDrive/..."
FEDAVG_SAVE_PATH = f"{SAVE_DIR}/MNIST_FedAvg_02.h5"
FEDAVG_METRICS_CSV = f"{SAVE_DIR}/FedAvg_metrics_rows_02.csv"
os.makedirs(SAVE_DIR, exist_ok=True)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------- Model Definition ----------------
def build_mnist_cnn():
    """
    Build the CNN used as the global model for FedAvg.
    """
    inp = keras.Input(shape=(28, 28, 1))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)

    feat = layers.Dense(128, activation="relu", name="feat")(x)
    logits = layers.Dense(NUM_CLASSES, name="logits")(feat)

    return keras.Model(inp, logits, name="mnist_cnn")


def clone_model_with_weights(model):
    """
    Create a copy of a model, including its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise average across client model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def eval_logits(model, ds):
    """
    Evaluate loss and accuracy for a model that outputs logits.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_full_ce(model, ds, lr=1e-3, batches_limit=None):
    """
    Perform local client training using standard cross-entropy loss.
    Returns the average local training loss and accuracy.
    """
    optimizer = tf.keras.optimizers.Adam(lr)
    acc = tf.keras.metrics.SparseCategoricalAccuracy()

    total_loss, total_n = 0.0, 0
    step_count = 0

    for x, y in ds:
        with tf.GradientTape() as tape:
            logits = model(x, training=True)
            loss = loss_fn(y, logits)

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


# ---------------- Early Stopping ----------------
class TrainAccEarlyStopping:
    """
    Stop training when the monitored training accuracy
    does not improve for a given number of rounds.
    """
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad = 0

    def step(self, train_acc_value):
        train_acc_value = float(train_acc_value)

        if self.best is None:
            self.best = train_acc_value
            self.bad = 0
            return False

        improved = (train_acc_value - self.best) > self.min_delta
        if improved:
            self.best = train_acc_value
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- FedAvg Training ----------------
server = build_mnist_cnn()
_ = server(tf.zeros([1, 28, 28, 1]))

# Metric lists kept explicitly for later export
FedAvg_Training_Loss = []
FedAvg_Training_Acc = []
FedAvg_Val_Loss = []
FedAvg_Val_Acc = []

# Round-level log dictionary
logs = {
    "round": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for r in range(ROUNDS):
    client_ws = []
    trL, trA, vaL, vaA = [], [], [], []

    for k in range(NUM_CLIENTS):
        local_model = clone_model_with_weights(server)

        local_train_loss, local_train_acc = local_train_full_ce(
            local_model,
            client_train_datasets[k],
            lr=LR_BACKBONE,
            batches_limit=BATCHES_PER_ROUND
        )

        local_val_loss, local_val_acc = eval_logits(
            local_model,
            client_val_datasets[k]
        )

        trL.append(local_train_loss)
        trA.append(local_train_acc)
        vaL.append(local_val_loss)
        vaA.append(local_val_acc)

        client_ws.append(local_model.get_weights())

    # Aggregate client models into the updated global model
    server.set_weights(average_weights(client_ws))

    # Compute mean round-level metrics across clients
    round_tr_loss = float(np.mean(trL))
    round_tr_acc = float(np.mean(trA))
    round_va_loss = float(np.mean(vaL))
    round_va_acc = float(np.mean(vaA))

    # Store metrics for logging and export
    logs["round"].append(r)
    logs["train_loss"].append(round_tr_loss)
    logs["train_acc"].append(round_tr_acc)
    logs["val_loss"].append(round_va_loss)
    logs["val_acc"].append(round_va_acc)

    FedAvg_Training_Loss.append(round_tr_loss)
    FedAvg_Training_Acc.append(round_tr_acc)
    FedAvg_Val_Loss.append(round_va_loss)
    FedAvg_Val_Acc.append(round_va_acc)

    print(
        f"[MNIST-FedAvg][Round {r:02d}] "
        f"Train loss={round_tr_loss:.4f}, acc={round_tr_acc:.4f} | "
        f"Val loss={round_va_loss:.4f}, acc={round_va_acc:.4f}"
    )

    # Stop if training accuracy has not improved
    if stopper.step(round_tr_acc):
        print(
            f"Early stopping at round {r:02d} "
            f"(no improvement in training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final Test Evaluation ----------------
test_loss, test_acc = eval_logits(server, test_ds)

print("\n[MNIST-FedAvg] Final test evaluation")
print(f"Test loss={test_loss:.4f}, Test acc={test_acc:.4f}")

# ---------------- Save Model ----------------
server.save(FEDAVG_SAVE_PATH)
print("\nSaved MNIST FedAvg global model to:", FEDAVG_SAVE_PATH)

# ---------------- Save Metrics ----------------
rows = [
    ["FedAvg_Training_Loss"] + FedAvg_Training_Loss,
    ["FedAvg_Training_Acc"] + FedAvg_Training_Acc,
    ["FedAvg_Val_Loss"] + FedAvg_Val_Loss,
    ["FedAvg_Val_Acc"] + FedAvg_Val_Acc,
]

with open(FEDAVG_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Saved FedAvg metrics CSV to:", FEDAVG_METRICS_CSV)

4: MHD-FL on MNIST

In [ ]:
# ============================================================
# Purpose:
# - Load the FedAvg model from Cell 3
# - Rebuild the shared backbone and global head
# - Train the backbone, global head, and personal head jointly on each client
# - Aggregate only the shared backbone and global head across clients
# - Keep the personal heads private to each client
# - Track training, validation, and KD-related metrics across rounds
# - Evaluate the final global and personalized models
# - Save the final global model and the combined personalized-head model
# ============================================================

import os
import json
import csv
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Required objects from Cell 1 ----------------
required_vars = [
    "client_train_datasets",
    "client_val_datasets",
    "test_ds",
    "client_present",
    "NUM_CLIENTS",
    "NUM_CLASSES",
    "BATCH_SIZE",
    "SEED",
]
for name in required_vars:
    assert name in globals(), f"Missing {name}. Run Cell 1 first."

SAVE_DIR = "/content/drive/MyDrive/....."
MDL_GLOBAL_PATH = f"{SAVE_DIR}/MDL_global_no_freeze_02.h5"
MDL_HEADS_DIR = f"{SAVE_DIR}/MDL_heads_no_freeze_h5"
MDL_LOGS_PATH = f"{SAVE_DIR}/MDL_logs_no_freeze_02.json"
MDL_METRICS_CSV = f"{SAVE_DIR}/MDL_metrics_no_freeze_rows_02.csv"
os.makedirs(MDL_HEADS_DIR, exist_ok=True)

# ---------------- Locate the FedAvg checkpoint ----------------
fedavg_candidates = (
    glob.glob(f"{SAVE_DIR}/MNIST_FedAvg_02.h5") +
    glob.glob(f"{SAVE_DIR}/*/MNIST_FedAvg_02*.h5") +
    glob.glob(f"{SAVE_DIR}/**/MNIST_FedAvg_02*.h5", recursive=True)
)

fedavg_candidates = sorted(set(fedavg_candidates))

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(f"No FedAvg checkpoint found in {SAVE_DIR}")

FEDAVG_PATH = next(
    (p for p in fedavg_candidates if os.path.basename(p) == "MNIST_FedAvg_02.h5"),
    fedavg_candidates[0]
)
print("Using FedAvg checkpoint:", FEDAVG_PATH)

# ---------------- Training configuration ----------------
ROUNDS = 10
BATCHES_PER_ROUND = None
LR_JOINT = 1e-3

TEMP = 2.0
L_G2P = 1.0
L_P2G = 1.0

EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def load_normalized_mnist_test_like_cell1():
    """
    Reconstruct the MNIST test set using the same preprocessing
    and normalization used in Cell 1.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_te = y_te.reshape(-1).astype("int32")
    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7

    x_te = (x_te - mean) / std
    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_test_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(set(client_present[client_id])), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def kd_kl(p_logits, q_logits, T=2.0):
    """
    KL divergence between softened prediction distributions.
    """
    p = tf.nn.softmax(p_logits / T, axis=-1)
    q = tf.nn.softmax(q_logits / T, axis=-1)

    kl = tf.reduce_mean(
        tf.reduce_sum(
            p * (tf.math.log(p + 1e-8) - tf.math.log(q + 1e-8)),
            axis=-1
        )
    )
    return kl * (T * T)


def apply_gradients_safe(optimizer, grads, variables):
    """
    Apply gradients after removing any None entries.
    """
    grads_and_vars = [(g, v) for g, v in zip(grads, variables) if g is not None]
    if grads_and_vars:
        optimizer.apply_gradients(grads_and_vars)


def eval_full_model(model, ds):
    """
    Evaluate a full end-to-end model on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def eval_head(backbone, head, ds):
    """
    Evaluate a backbone-head pair on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        features = backbone(x, training=False)
        logits = head(features, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_mdl_no_freeze(local_backbone, local_global_head, personal_head, ds,
                              lr=1e-3, batches_limit=None):
    """
    Joint local update for the no-freeze variant.

    The backbone, local global head, and personal head are all updated together.

    Objective:
        CE(global) + CE(personal) + KD(global -> personal) + KD(personal -> global)
    """
    optimizer = tf.keras.optimizers.Adam(lr)

    acc_g = tf.keras.metrics.SparseCategoricalAccuracy()
    acc_p = tf.keras.metrics.SparseCategoricalAccuracy()

    total_ce_g = 0.0
    total_ce_p = 0.0
    total_kd_g2p = 0.0
    total_kd_p2g = 0.0
    total_n = 0
    step_count = 0

    for x, y in ds:
        with tf.GradientTape() as tape:
            features = local_backbone(x, training=True)
            logits_global = local_global_head(features, training=True)
            logits_personal = personal_head(features, training=True)

            ce_global = loss_fn(y, logits_global)
            ce_personal = loss_fn(y, logits_personal)

            kd_g2p = kd_kl(logits_global, logits_personal, T=TEMP)
            kd_p2g = kd_kl(logits_personal, logits_global, T=TEMP)

            loss = (
                ce_global
                + ce_personal
                + (L_G2P * kd_g2p)
                + (L_P2G * kd_p2g)
            )

        variables = (
            local_backbone.trainable_variables
            + local_global_head.trainable_variables
            + personal_head.trainable_variables
        )
        grads = tape.gradient(loss, variables)
        apply_gradients_safe(optimizer, grads, variables)

        bs = int(tf.shape(x)[0].numpy())
        total_ce_g += float(ce_global.numpy()) * bs
        total_ce_p += float(ce_personal.numpy()) * bs
        total_kd_g2p += float(kd_g2p.numpy()) * bs
        total_kd_p2g += float(kd_p2g.numpy()) * bs
        total_n += bs

        acc_g.update_state(y, tf.nn.softmax(logits_global))
        acc_p.update_state(y, tf.nn.softmax(logits_personal))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return (
        float(total_ce_g / max(total_n, 1)),
        float(acc_g.result().numpy()),
        float(total_ce_p / max(total_n, 1)),
        float(acc_p.result().numpy()),
        float(total_kd_g2p / max(total_n, 1)),
        float(total_kd_p2g / max(total_n, 1)),
    )


class TrainAccEarlyStopping:
    """
    Stop training when the monitored training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad_rounds = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad_rounds = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad_rounds = 0
        else:
            self.bad_rounds += 1

        return self.bad_rounds >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the FedAvg model ----------------
fedavg_model = keras.models.load_model(FEDAVG_PATH, compile=False)
print("Loaded FedAvg model.")

# Reconstruct the shared backbone and global head from the FedAvg checkpoint
backbone = keras.Model(
    fedavg_model.input,
    fedavg_model.get_layer("feat").output,
    name="backbone"
)
feat_dim = backbone.output_shape[-1]

server_global_head = layers.Dense(NUM_CLASSES, name="server_global_head")
server_global_head(tf.zeros([1, feat_dim]))
server_global_head.set_weights(fedavg_model.get_layer("logits").get_weights())

# Confirm that the reconstructed backbone/head path matches the original model
init_loss_full, init_acc_full = eval_full_model(fedavg_model, test_ds)
init_loss_split, init_acc_split = eval_head(backbone, server_global_head, test_ds)

print("\nInitial consistency check")
print(f"Full FedAvg model:       loss={init_loss_full:.4f}, acc={init_acc_full:.4f}")
print(f"Backbone + global head:  loss={init_loss_split:.4f}, acc={init_acc_split:.4f}")

if abs(init_loss_full - init_loss_split) > 1e-5 or abs(init_acc_full - init_acc_split) > 1e-5:
    raise ValueError(
        "Mismatch between the loaded FedAvg model and the reconstructed backbone/head path."
    )

# ---------------- Initialize client-specific heads ----------------
personal_heads = []
for client_id in range(NUM_CLIENTS):
    head = layers.Dense(NUM_CLASSES, name=f"personal_head_{client_id}")
    head(tf.zeros([1, feat_dim]))
    head.set_weights(server_global_head.get_weights())
    personal_heads.append(head)

# Round-level metric storage
logs = {
    "round": [],
    "train_loss_global": [],
    "train_acc_global": [],
    "train_loss_personal": [],
    "train_acc_personal": [],
    "val_loss_global": [],
    "val_acc_global": [],
    "val_loss_personal": [],
    "val_acc_personal": [],
    "kd_g2p": [],
    "kd_p2g": [],
}

# ---------------- Federated training loop ----------------
for r in range(ROUNDS):
    client_backbone_ws = []
    client_global_ws = []

    trLg, trAg = [], []
    trLp, trAp = [], []
    vaLg, vaAg = [], []
    vaLp, vaAp = [], []
    kd_g2p_vals, kd_p2g_vals = [], []

    for client_id in range(NUM_CLIENTS):
        local_backbone = clone_model_with_weights(backbone)

        local_global_head = layers.Dense(NUM_CLASSES, name=f"local_global_head_{client_id}")
        local_global_head(tf.zeros([1, feat_dim]))
        local_global_head.set_weights(server_global_head.get_weights())

        l_g, a_g, l_p, a_p, kd_g2p, kd_p2g = local_train_mdl_no_freeze(
            local_backbone=local_backbone,
            local_global_head=local_global_head,
            personal_head=personal_heads[client_id],
            ds=client_train_datasets[client_id],
            lr=LR_JOINT,
            batches_limit=BATCHES_PER_ROUND,
        )

        lvg, avg = eval_head(local_backbone, local_global_head, client_val_datasets[client_id])
        lvp, avp = eval_head(local_backbone, personal_heads[client_id], client_val_datasets[client_id])

        trLg.append(l_g)
        trAg.append(a_g)
        trLp.append(l_p)
        trAp.append(a_p)

        vaLg.append(lvg)
        vaAg.append(avg)
        vaLp.append(lvp)
        vaAp.append(avp)

        kd_g2p_vals.append(kd_g2p)
        kd_p2g_vals.append(kd_p2g)

        client_backbone_ws.append(local_backbone.get_weights())
        client_global_ws.append(local_global_head.get_weights())

    # Aggregate only the shared components
    backbone.set_weights(average_weights(client_backbone_ws))
    server_global_head.set_weights(average_weights(client_global_ws))

    round_tr_lg = float(np.mean(trLg))
    round_tr_ag = float(np.mean(trAg))
    round_tr_lp = float(np.mean(trLp))
    round_tr_ap = float(np.mean(trAp))

    round_va_lg = float(np.mean(vaLg))
    round_va_ag = float(np.mean(vaAg))
    round_va_lp = float(np.mean(vaLp))
    round_va_ap = float(np.mean(vaAp))

    round_kd_g2p = float(np.mean(kd_g2p_vals))
    round_kd_p2g = float(np.mean(kd_p2g_vals))

    logs["round"].append(r)
    logs["train_loss_global"].append(round_tr_lg)
    logs["train_acc_global"].append(round_tr_ag)
    logs["train_loss_personal"].append(round_tr_lp)
    logs["train_acc_personal"].append(round_tr_ap)
    logs["val_loss_global"].append(round_va_lg)
    logs["val_acc_global"].append(round_va_ag)
    logs["val_loss_personal"].append(round_va_lp)
    logs["val_acc_personal"].append(round_va_ap)
    logs["kd_g2p"].append(round_kd_g2p)
    logs["kd_p2g"].append(round_kd_p2g)

    print(
        f"[MDL no-freeze][Round {r:02d}] "
        f"G train: loss={round_tr_lg:.3f}, acc={round_tr_ag:.3f} | "
        f"P train: loss={round_tr_lp:.3f}, acc={round_tr_ap:.3f} || "
        f"G val: loss={round_va_lg:.3f}, acc={round_va_ag:.3f} | "
        f"P val: loss={round_va_lp:.3f}, acc={round_va_ap:.3f} | "
        f"KD g->p={round_kd_g2p:.4f}, p->g={round_kd_p2g:.4f}"
    )

    if stopper.step(round_tr_ap):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in personal training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
global_test_loss, global_test_acc = eval_head(backbone, server_global_head, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)

    l_spec, a_spec = eval_head(backbone, personal_heads[client_id], spec_ds_k)
    l_glob, a_glob = eval_head(backbone, personal_heads[client_id], test_ds)

    spec_losses.append(l_spec)
    spec_accs.append(a_spec)
    glob_losses.append(l_glob)
    glob_accs.append(a_glob)

print("\nFinal results")
print(f"Global model on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save the final models ----------------
inp = keras.Input(shape=fedavg_model.input_shape[1:])
out = server_global_head(backbone(inp))

global_model = keras.Model(inp, out, name="global_HD_FL")
GLOBAL_MODEL_PATH = os.path.join(SAVE_DIR, "global_HD-FL.h5")
global_model.save(GLOBAL_MODEL_PATH)

print("\nSaved global model to:", GLOBAL_MODEL_PATH)

inputs = keras.Input(shape=(feat_dim,))
outputs = []
for head in personal_heads:
    outputs.append(head(inputs))

personal_model = keras.Model(inputs, outputs, name="personal_HD_FL")
PERSONAL_MODEL_PATH = os.path.join(SAVE_DIR, "person_HD-FL.h5")
personal_model.save(PERSONAL_MODEL_PATH)

print("Saved personalized heads model to:", PERSONAL_MODEL_PATH)

3: Baseline 1 LL-FT Personalization on MNIST

In [ ]:
# ===========================================================
# Purpose:
# - Use the client datasets created in Cell 1
# - Load the FedAvg model saved in Cell 2
# - Freeze the shared backbone and train only a private classifier head for each client
# - Apply early stopping during head training based on training accuracy
# - Evaluate each personalized head on:
#     (A) a specialization test set containing only the labels available to that client
#     (B) the full MNIST test set
# - Report per-client results and overall averages
# - Save the frozen global model and the personalized heads
#
# Saves:
#   LLFT_global_02.h5
#   LLFT_personal_heads_02.npz
# ============================================================

import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Required objects from Cell 1 ----------------
assert "client_train_datasets" in globals()
assert "client_present" in globals()
assert "NUM_CLIENTS" in globals()
assert "NUM_CLASSES" in globals()
assert "BATCH_SIZE" in globals()
assert "SEED" in globals()

SAVE_DIR = "/content/drive/MyDrive/....."

# Locate the FedAvg checkpoint saved in Cell 2
fedavg_candidates = glob.glob(f"{SAVE_DIR}/*MNIST_FedAvg_02.h5")
print("FedAvg candidates:", fedavg_candidates)

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(
        f"No FedAvg checkpoint found in {SAVE_DIR}. "
        f"Current contents: {os.listdir(SAVE_DIR)}"
    )

FEDAVG_PATH = fedavg_candidates[0]
print("Using FedAvg checkpoint:", FEDAVG_PATH)

LLFT_GLOBAL_PATH = f"{SAVE_DIR}/LLFT_global_02.h5"
LLFT_PERSONAL_PATH = f"{SAVE_DIR}/LLFT_personal_heads_02.npz"

# ---------------- Personalization configuration ----------------
EPOCHS_HEAD = 3
LR_HEAD = 2e-3
BATCHES_LIMIT = None

# Early stopping during head-only training
EARLYSTOP_PATIENCE = 2
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------- Rebuild the normalized MNIST test arrays ----------------
# This follows the same preprocessing used in Cell 1 so that
# specialization test subsets are defined consistently.
def load_normalized_mnist_arrays_like_cell1():
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_tr = y_tr.reshape(-1).astype("int32")
    y_te = y_te.reshape(-1).astype("int32")

    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7

    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    return x_tr, y_tr, x_te, y_te


_, _, x_test_arr, y_test_arr = load_normalized_mnist_arrays_like_cell1()


def make_test_ds_for_labels(x_te, y_te, allowed_labels, batch_size):
    """
    Create a filtered test dataset containing only the specified labels.
    """
    allowed = np.array(sorted(list(allowed_labels)), dtype=np.int32)
    mask = np.isin(y_te, allowed)

    x = x_te[mask]
    y = y_te[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


# ---------------- Evaluation helper ----------------
def eval_head(backbone, head, ds):
    """
    Evaluate a frozen backbone paired with a classification head.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        features = backbone(x, training=False)
        logits = head(features, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


# ---------------- Early stopping helper ----------------
class TrainAccEarlyStopping:
    """
    Stop local head training when training accuracy
    no longer improves for a fixed number of epochs.
    """
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad = 0

    def step(self, train_acc_value):
        train_acc_value = float(train_acc_value)

        if self.best is None:
            self.best = train_acc_value
            self.bad = 0
            return False

        improved = (train_acc_value - self.best) > self.min_delta
        if improved:
            self.best = train_acc_value
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


def train_head_only(backbone, head, ds, epochs=1, lr=2e-3, batches_limit=None,
                    earlystop_patience=2, earlystop_min_delta=0.0):
    """
    Train only the client-specific head while keeping the backbone frozen.
    Early stopping is based on training accuracy.
    """
    optimizer = tf.keras.optimizers.Adam(lr)
    stopper = TrainAccEarlyStopping(
        patience=earlystop_patience,
        min_delta=earlystop_min_delta
    )

    for epoch in range(epochs):
        acc = tf.keras.metrics.SparseCategoricalAccuracy()
        total_loss, total_n = 0.0, 0
        step_count = 0

        for x, y in ds:
            features = tf.stop_gradient(backbone(x, training=False))

            with tf.GradientTape() as tape:
                logits = head(features, training=True)
                loss = loss_fn(y, logits)

            grads = tape.gradient(loss, head.trainable_variables)
            optimizer.apply_gradients(zip(grads, head.trainable_variables))

            bs = int(tf.shape(x)[0].numpy())
            total_loss += float(loss.numpy()) * bs
            total_n += bs
            acc.update_state(y, tf.nn.softmax(logits))

            step_count += 1
            if batches_limit is not None and step_count >= batches_limit:
                break

        train_acc_epoch = float(acc.result().numpy())
        if stopper.step(train_acc_epoch):
            break


# ---------------- Load FedAvg and define the model components ----------------
fedavg_model = keras.models.load_model(FEDAVG_PATH)
print("Loaded FedAvg model.")

# Reconstruct the shared feature extractor from the FedAvg model
backbone = keras.Model(
    fedavg_model.input,
    fedavg_model.get_layer("feat").output,
    name="backbone"
)
backbone.trainable = False
feat_dim = backbone.output_shape[-1]

# Reconstruct the global classifier head from the FedAvg model
global_head = layers.Dense(NUM_CLASSES, name="global_head")
global_head(tf.zeros([1, feat_dim]))
global_head.set_weights(fedavg_model.get_layer("logits").get_weights())
global_head.trainable = False

# Initialize one private head per client from the global head weights
personal_heads = []
for client_id in range(NUM_CLIENTS):
    head = layers.Dense(NUM_CLASSES, name=f"personal_head_{client_id}")
    head(tf.zeros([1, feat_dim]))
    head.set_weights(global_head.get_weights())
    personal_heads.append(head)

# ---------------- Baseline evaluation before personalization ----------------
base_global_loss, base_global_acc = eval_head(backbone, global_head, test_ds)
print(f"\n[Baseline global head on full test set] loss={base_global_loss:.4f}, acc={base_global_acc:.4f}")

# Evaluate the frozen global head on each client's specialization test set
base_spec_accs = []
for client_id in range(NUM_CLIENTS):
    spec_ds_k, n_k = make_test_ds_for_labels(
        x_test_arr, y_test_arr, set(client_present[client_id]), BATCH_SIZE
    )

    if n_k == 0:
        base_spec_accs.append(np.nan)
        continue

    _, acc_value = eval_head(backbone, global_head, spec_ds_k)
    base_spec_accs.append(acc_value)

print(f"[Baseline global head average specialization accuracy] {float(np.nanmean(base_spec_accs)):.4f}")

# ---------------- Personalize each client head and evaluate ----------------
spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    print(f"\n=== Client {client_id:02d} head personalization ===")
    print("Available labels:", client_present[client_id])

    # Train only the private head for this client
    train_head_only(
        backbone=backbone,
        head=personal_heads[client_id],
        ds=client_train_datasets[client_id],
        epochs=EPOCHS_HEAD,
        lr=LR_HEAD,
        batches_limit=BATCHES_LIMIT,
        earlystop_patience=EARLYSTOP_PATIENCE,
        earlystop_min_delta=EARLYSTOP_MIN_DELTA
    )

    # Evaluate on the client-specific specialization test set
    spec_ds_k, n_spec = make_test_ds_for_labels(
        x_test_arr, y_test_arr, set(client_present[client_id]), BATCH_SIZE
    )
    spec_loss, spec_acc = eval_head(backbone, personal_heads[client_id], spec_ds_k)

    # Evaluate on the full global test set
    glob_loss, glob_acc = eval_head(backbone, personal_heads[client_id], test_ds)

    spec_losses.append(spec_loss)
    spec_accs.append(spec_acc)
    glob_losses.append(glob_loss)
    glob_accs.append(glob_acc)

    print(f"  Specialization test: n={n_spec:5d}  loss={spec_loss:.4f}  acc={spec_acc:.4f}")
    print(f"  Global test:               loss={glob_loss:.4f}  acc={glob_acc:.4f}")

print("\n==================== Average Performance Across Clients ====================")
print(f"Average specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average global test:         loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save outputs ----------------
# Save the frozen global baseline model
inp = fedavg_model.input
out = global_head(backbone(inp))
llft_global_model = keras.Model(inp, out, name="LLFT_global_model")
llft_global_model.save(LLFT_GLOBAL_PATH)

# Save all personalized heads in a single NPZ file
personal_weights = {
    f"client_{client_id}": np.array(personal_heads[client_id].get_weights(), dtype=object)
    for client_id in range(NUM_CLIENTS)
}
np.savez(LLFT_PERSONAL_PATH, **personal_weights)

print("\nSaved files:")
print(" -", LLFT_GLOBAL_PATH)
print(" -", LLFT_PERSONAL_PATH)

5: Baseline 2 FedGKD On MNIST

In [ ]:
# ============================================================
# Purpose:
# - Use the client datasets created in Cell 1
# - Load the FedAvg model from Cell 2 as a fixed teacher
# - Train a full student model on each client using cross-entropy and KD
# - Aggregate student models across clients using FedAvg
# - Apply early stopping based on average training accuracy across clients
# - Track round-level training, validation, and KD metrics
# - Evaluate:
#     (1) the final global server model on the full test set
#     (2) the final client models on their specialization test sets
#     (3) the final client models on the full test set
# - Save the final global KD model and metric logs
#
# Saves:
#   KD_Global.h5
#   KD_logs.json
#   KD_metrics_rows.csv
# ============================================================

import os
import glob
import json
import csv
import numpy as np
import tensorflow as tf
from tensorflow import keras

# ---------------- Required objects from Cell 1 ----------------
assert "client_train_datasets" in globals()
assert "client_val_datasets" in globals()
assert "test_ds" in globals()
assert "client_present" in globals()
assert "NUM_CLIENTS" in globals()
assert "NUM_CLASSES" in globals()
assert "BATCH_SIZE" in globals()
assert "SEED" in globals()

SAVE_DIR = "/content/drive/MyDrive/...."
KD_GLOBAL_SAVE_PATH = f"{SAVE_DIR}/KD_Global.h5"
KD_LOGS_PATH = f"{SAVE_DIR}/KD_logs.json"
KD_METRICS_CSV = f"{SAVE_DIR}/KD_metrics_rows.csv"

# ---------------- Locate the FedAvg teacher checkpoint ----------------
fedavg_candidates = glob.glob(f"{SAVE_DIR}/*MNIST_FedAvg*.h5")
print("FedAvg candidates:", fedavg_candidates)

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(
        f"No FedAvg checkpoint found in {SAVE_DIR}. "
        f"Current contents: {os.listdir(SAVE_DIR)}"
    )

FEDAVG_PATH = fedavg_candidates[0]
print("Using teacher checkpoint:", FEDAVG_PATH)

# ---------------- Knowledge distillation configuration ----------------
ROUNDS = 10
LR_KD = 1e-3
BATCHES_PER_ROUND = None

TEMP = 2.0
L_G2P = 1.0

# Early stopping is based on round-level training accuracy
EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------- Rebuild the normalized MNIST test arrays ----------------
# This follows the same preprocessing used in Cell 1 so that
# client-specific specialization test sets are constructed consistently.
def load_normalized_mnist_test_like_cell1():
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_tr = y_tr.reshape(-1).astype("int32")
    y_te = y_te.reshape(-1).astype("int32")

    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7
    x_te = (x_te - mean) / std

    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_test_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(list(set(client_present[client_id]))), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


# ---------------- Helper functions ----------------
def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def kd_kl(teacher_logits, student_logits, T=2.0):
    """
    Compute the KL divergence between softened teacher and student predictions.
    """
    teacher_probs = tf.nn.softmax(teacher_logits / T, axis=-1)
    student_probs = tf.nn.softmax(student_logits / T, axis=-1)

    kl = tf.reduce_mean(
        tf.reduce_sum(
            teacher_probs * (
                tf.math.log(teacher_probs + 1e-8) -
                tf.math.log(student_probs + 1e-8)
            ),
            axis=-1
        )
    )
    return kl * (T * T)


def eval_logits(model, ds):
    """
    Evaluate a model that returns logits.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_model_level_kd(teacher_fixed, student, ds, lr=1e-3, batches_limit=None):
    """
    Train a student model locally using cross-entropy and
    model-level knowledge distillation from a fixed teacher.
    """
    optimizer = tf.keras.optimizers.Adam(lr)
    acc = tf.keras.metrics.SparseCategoricalAccuracy()

    total_loss, total_kd, total_n = 0.0, 0.0, 0
    step_count = 0

    for x, y in ds:
        with tf.GradientTape() as tape:
            teacher_logits = teacher_fixed(x, training=False)
            student_logits = student(x, training=True)

            ce = loss_fn(y, student_logits)
            kd = kd_kl(teacher_logits, student_logits, T=TEMP)
            loss = ce + (L_G2P * kd)

        grads = tape.gradient(loss, student.trainable_variables)
        optimizer.apply_gradients(zip(grads, student.trainable_variables))

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_kd += float(kd.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(student_logits))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return (
        float(total_loss / max(total_n, 1)),
        float(acc.result().numpy()),
        float(total_kd / max(total_n, 1))
    )


# ---------------- Early stopping ----------------
class TrainAccEarlyStopping:
    """
    Stop training when the monitored training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the fixed teacher model ----------------
teacher = keras.models.load_model(FEDAVG_PATH)
dummy = tf.zeros([1] + list(teacher.input_shape[1:]), dtype=tf.float32)
_ = teacher(dummy, training=False)
print("Loaded fixed teacher model.")

# Initialize the server model from the teacher weights
kd_server = clone_model_with_weights(teacher)

# Metric storage for JSON and CSV export
logs = {
    "round": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "kd": []
}

# ---------------- Federated KD training loop ----------------
last_round_students = None

for r in range(ROUNDS):
    client_ws = []
    trL, trA, vaL, vaA, kd_vals = [], [], [], [], []
    students_this_round = []

    for client_id in range(NUM_CLIENTS):
        # Each client starts from the current server model
        student = clone_model_with_weights(kd_server)

        local_train_loss, local_train_acc, local_kd = local_train_model_level_kd(
            teacher_fixed=teacher,
            student=student,
            ds=client_train_datasets[client_id],
            lr=LR_KD,
            batches_limit=BATCHES_PER_ROUND
        )

        local_val_loss, local_val_acc = eval_logits(student, client_val_datasets[client_id])

        trL.append(local_train_loss)
        trA.append(local_train_acc)
        vaL.append(local_val_loss)
        vaA.append(local_val_acc)
        kd_vals.append(local_kd)

        client_ws.append(student.get_weights())
        students_this_round.append(student)

    # Aggregate student models using FedAvg
    kd_server.set_weights(average_weights(client_ws))

    round_tr_loss = float(np.mean(trL))
    round_tr_acc = float(np.mean(trA))
    round_va_loss = float(np.mean(vaL))
    round_va_acc = float(np.mean(vaA))
    round_kd = float(np.mean(kd_vals))

    logs["round"].append(r)
    logs["train_loss"].append(round_tr_loss)
    logs["train_acc"].append(round_tr_acc)
    logs["val_loss"].append(round_va_loss)
    logs["val_acc"].append(round_va_acc)
    logs["kd"].append(round_kd)

    print(
        f"[Model-KD fixed teacher][Round {r:02d}] "
        f"Train loss={round_tr_loss:.3f}, acc={round_tr_acc:.3f} | "
        f"Val loss={round_va_loss:.3f}, acc={round_va_acc:.3f} | "
        f"KD={round_kd:.4f}"
    )

    # Keep the final client models from the last completed round
    last_round_students = students_this_round

    if stopper.step(round_tr_acc):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
# (1) Global server model on the full test set
# (2) Final client models on specialization test sets
# (3) Final client models on the full test set
global_test_loss, global_test_acc = eval_logits(kd_server, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)

    spec_loss, spec_acc = eval_logits(last_round_students[client_id], spec_ds_k)
    glob_loss, glob_acc = eval_logits(last_round_students[client_id], test_ds)

    spec_losses.append(spec_loss)
    spec_accs.append(spec_acc)
    glob_losses.append(glob_loss)
    glob_accs.append(glob_acc)

print("\nFinal results")
print(f"Global server on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save model and logs ----------------
kd_server.save(KD_GLOBAL_SAVE_PATH)
print("\nSaved KD global model to:", KD_GLOBAL_SAVE_PATH)

with open(KD_LOGS_PATH, "w") as f:
    json.dump(logs, f, indent=2)
print("Saved KD logs to:", KD_LOGS_PATH)

rows = [
    ["round"] + logs["round"],
    ["train_loss"] + logs["train_loss"],
    ["train_acc"] + logs["train_acc"],
    ["val_loss"] + logs["val_loss"],
    ["val_acc"] + logs["val_acc"],
    ["kd"] + logs["kd"],
]

with open(KD_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Saved KD metrics CSV to:", KD_METRICS_CSV)

6: Baseline 3 FedPSD on MNIST

In [ ]:
# ============================================================
# Purpose:
# - Use the client datasets created in Cell 1
# - Initialize the global model from the FedAvg checkpoint saved in Cell 3
# - Train each client model with progressive self-distillation and logits calibration
# - Aggregate client models with FedAvg after each round
# - Apply early stopping based on average training accuracy across clients
# - Track round-level training, validation, and KD metrics
# - Evaluate:
#     (1) the final global server model on the full test set
#     (2) the final client models on their specialization test sets
#     (3) the final client models on the full test set
# - Save the final global model and the metric logs
#
# Saves:
#   FedPSD_Global.h5
#   FedPSD_logs.json
#   FedPSD_metrics_rows.csv
# ============================================================

import os
import glob
import json
import csv
import numpy as np
import tensorflow as tf
from tensorflow import keras

# ---------------- Required objects from Cell 1 ----------------
for name in [
    "client_train_datasets",
    "client_val_datasets",
    "test_ds",
    "client_present",
    "client_idxs",
    "y_train",
    "NUM_CLIENTS",
    "NUM_CLASSES",
    "BATCH_SIZE",
    "SEED",
]:
    assert name in globals(), f"Missing {name}. Run Cell 1 first."

SAVE_DIR = "/content/drive/MyDrive/...."
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------- Locate the FedAvg initialization checkpoint ----------------
fedavg_candidates = glob.glob(f"{SAVE_DIR}/*MNIST_FedAvg_02*.h5")
print("FedAvg candidates:", fedavg_candidates)

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(
        f"No FedAvg checkpoint found in {SAVE_DIR}. "
        f"Current contents: {os.listdir(SAVE_DIR)}"
    )

FEDAVG_PATH = fedavg_candidates[0]
print("Using FedAvg initialization:", FEDAVG_PATH)

FEDPSD_GLOBAL_SAVE_PATH = f"{SAVE_DIR}/FedPSD_Global.h5"
FEDPSD_LOGS_PATH = f"{SAVE_DIR}/FedPSD_logs.json"
FEDPSD_METRICS_CSV = f"{SAVE_DIR}/FedPSD_metrics_rows.csv"

# ---------------- Training configuration ----------------
ROUNDS = 10
LOCAL_EPOCHS = 2
BATCHES_PER_ROUND = None
LR_LOCAL = 1e-3

TEMP = 2.0
L_KD = 1.0
BETA_CAL = 1.0

# Early stopping is based on round-level training accuracy
EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

# Weight assigned to teacher soft targets increases over rounds
def gamma_round(r, R):
    return float(r) / float(max(R - 1, 1))


loss_ce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------- Rebuild the normalized MNIST test arrays ----------------
# This follows the same preprocessing used in Cell 1 so that
# specialization test sets are constructed consistently.
def load_normalized_mnist_test_like_cell1():
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_tr = y_tr.reshape(-1).astype("int32")
    y_te = y_te.reshape(-1).astype("int32")

    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7

    x_te = (x_te - mean) / std
    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_test_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(list(set(client_present[client_id]))), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


# ---------------- Helper functions ----------------
def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def kl_teacher_student(teacher_probs, student_logits, T=2.0):
    """
    Compute KL divergence between teacher probabilities and
    temperature-scaled student predictions.
    """
    student_probs = tf.nn.softmax(student_logits / T, axis=-1)
    teacher_probs = tf.clip_by_value(teacher_probs, 1e-8, 1.0)
    student_probs = tf.clip_by_value(student_probs, 1e-8, 1.0)

    kl = tf.reduce_mean(
        tf.reduce_sum(
            teacher_probs * (tf.math.log(teacher_probs) - tf.math.log(student_probs)),
            axis=-1
        )
    )
    return kl * (T * T)


def eval_logits(model, ds):
    """
    Evaluate a model that returns logits.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_ce(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def client_prior_pi_from_idxs(client_id):
    """
    Estimate the empirical class prior for one client.
    """
    idx = np.array(client_idxs[client_id], dtype=np.int64)
    y_client = y_train[idx]

    counts = np.bincount(y_client, minlength=NUM_CLASSES).astype(np.float32)
    pi = counts / max(counts.sum(), 1.0)
    pi = np.clip(pi, 1e-8, 1.0)
    return pi


# ---------------- Local FedPSD training ----------------
# Returns:
#   train_loss, train_acc, kd_avg
def local_train_fedpsd(
    student_model,
    ds,
    pi_k,
    r,
    R,
    local_epochs=2,
    lr=1e-3,
    batches_limit=None,
    personal_teacher_model=None,
):
    """
    Train a full client model using progressive self-distillation
    and client-specific logits calibration.

    In the first local epoch, the teacher can be the previous-round
    personalized model. In later local epochs, the teacher becomes
    the model snapshot from the previous epoch.
    """
    optimizer = tf.keras.optimizers.Adam(lr)
    g = gamma_round(r, R)

    prev_epoch_teacher = None

    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_kd, total_n = 0.0, 0.0, 0

    log_pi = tf.constant(np.log(pi_k).astype(np.float32))

    for epoch in range(local_epochs):
        if epoch == 0:
            teacher_model = personal_teacher_model
        else:
            teacher_model = prev_epoch_teacher

        prev_epoch_teacher = clone_model_with_weights(student_model)

        step_count = 0
        for x, y in ds:
            y_onehot = tf.one_hot(y, depth=NUM_CLASSES)

            if teacher_model is None:
                teacher_probs = y_onehot
            else:
                teacher_logits = teacher_model(x, training=False)
                teacher_soft = tf.nn.softmax(teacher_logits, axis=-1)
                teacher_probs = (1.0 - g) * y_onehot + g * teacher_soft

            with tf.GradientTape() as tape:
                student_logits = student_model(x, training=True)
                calibrated_logits = student_logits + BETA_CAL * log_pi

                ce = loss_ce(y, calibrated_logits)
                kd = kl_teacher_student(teacher_probs, student_logits, T=TEMP)
                loss = ce + (L_KD * kd)

            grads = tape.gradient(loss, student_model.trainable_variables)
            optimizer.apply_gradients(zip(grads, student_model.trainable_variables))

            bs = int(tf.shape(x)[0].numpy())
            total_loss += float(loss.numpy()) * bs
            total_kd += float(kd.numpy()) * bs
            total_n += bs
            acc.update_state(y, tf.nn.softmax(student_logits))

            step_count += 1
            if batches_limit is not None and step_count >= batches_limit:
                break

    return (
        float(total_loss / max(total_n, 1)),
        float(acc.result().numpy()),
        float(total_kd / max(total_n, 1)),
    )


# ---------------- Early stopping ----------------
class TrainAccEarlyStopping:
    """
    Stop training when the monitored training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the FedAvg model as initialization ----------------
server = keras.models.load_model(FEDAVG_PATH)
dummy = tf.zeros([1] + list(server.input_shape[1:]), dtype=tf.float32)
_ = server(dummy, training=False)
print("Loaded server initialization model.")

personal_teachers = [None for _ in range(NUM_CLIENTS)]
last_round_students = None

# Metric storage for JSON and CSV export
logs = {
    "round": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "kd": [],
}

# ---------------- Federated FedPSD training loop ----------------
for r in range(ROUNDS):
    client_ws = []
    trL, trA, vaL, vaA, kd_vals = [], [], [], [], []
    students_this_round = []

    for client_id in range(NUM_CLIENTS):
        student = clone_model_with_weights(server)
        pi_k = client_prior_pi_from_idxs(client_id)

        local_train_loss, local_train_acc, local_kd = local_train_fedpsd(
            student_model=student,
            ds=client_train_datasets[client_id],
            pi_k=pi_k,
            r=r,
            R=ROUNDS,
            local_epochs=LOCAL_EPOCHS,
            lr=LR_LOCAL,
            batches_limit=BATCHES_PER_ROUND,
            personal_teacher_model=personal_teachers[client_id],
        )

        personal_teachers[client_id] = clone_model_with_weights(student)

        local_val_loss, local_val_acc = eval_logits(student, client_val_datasets[client_id])

        trL.append(local_train_loss)
        trA.append(local_train_acc)
        vaL.append(local_val_loss)
        vaA.append(local_val_acc)
        kd_vals.append(local_kd)

        client_ws.append(student.get_weights())
        students_this_round.append(student)

    server.set_weights(average_weights(client_ws))

    round_tr_loss = float(np.mean(trL))
    round_tr_acc = float(np.mean(trA))
    round_va_loss = float(np.mean(vaL))
    round_va_acc = float(np.mean(vaA))
    round_kd = float(np.mean(kd_vals))

    logs["round"].append(r)
    logs["train_loss"].append(round_tr_loss)
    logs["train_acc"].append(round_tr_acc)
    logs["val_loss"].append(round_va_loss)
    logs["val_acc"].append(round_va_acc)
    logs["kd"].append(round_kd)

    print(
        f"[FedPSD][Round {r:02d}] "
        f"Train loss={round_tr_loss:.3f} acc={round_tr_acc:.3f} | "
        f"Val loss={round_va_loss:.3f} acc={round_va_acc:.3f} | "
        f"KD={round_kd:.4f}"
    )

    last_round_students = students_this_round

    if stopper.step(round_tr_acc):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
global_test_loss, global_test_acc = eval_logits(server, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)
    spec_loss, spec_acc = eval_logits(last_round_students[client_id], spec_ds_k)
    glob_loss, glob_acc = eval_logits(last_round_students[client_id], test_ds)

    spec_losses.append(spec_loss)
    spec_accs.append(spec_acc)
    glob_losses.append(glob_loss)
    glob_accs.append(glob_acc)

print("\nFinal results")
print(f"Global server on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save model and logs ----------------
server.save(FEDPSD_GLOBAL_SAVE_PATH)
print("\nSaved FedPSD global model to:", FEDPSD_GLOBAL_SAVE_PATH)

with open(FEDPSD_LOGS_PATH, "w") as f:
    json.dump(logs, f, indent=2)
print("Saved FedPSD logs to:", FEDPSD_LOGS_PATH)

rows = [
    ["round"] + logs["round"],
    ["train_loss"] + logs["train_loss"],
    ["train_acc"] + logs["train_acc"],
    ["val_loss"] + logs["val_loss"],
    ["val_acc"] + logs["val_acc"],
    ["kd"] + logs["kd"],
]

with open(FEDPSD_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Saved FedPSD metrics CSV to:", FEDPSD_METRICS_CSV)

7: Baseline 4 Ditto Personalization on MNIST

In [ ]:
# ============================================================
# Purpose:
# - Initialize the global model from the FedAvg checkpoint saved in Cell 3
# - Maintain one personalized model per client
# - Update the global model with standard local cross-entropy training
# - Update each personalized model with the Ditto proximal objective
# - Apply early stopping based on average personalized training accuracy
# - Track round-level global and personalized training and validation metrics
# - Evaluate:
#     (1) the final global server model on the full test set
#     (2) the final personalized models on their specialization test sets
#     (3) the final personalized models on the full test set
# - Save the final global model, personalized models, and metric logs
#
# Saves:
#   Ditto_Global.h5
#   Ditto personal models
#   Ditto_logs.json
#   Ditto_metrics_rows.csv
# ============================================================

import os
import json
import glob
import csv
import numpy as np
import tensorflow as tf
from tensorflow import keras

# ---------------- Required objects from Cell 1 ----------------
for name in [
    "client_train_datasets",
    "client_val_datasets",
    "test_ds",
    "client_present",
    "client_idxs",
    "y_train",
    "NUM_CLIENTS",
    "NUM_CLASSES",
    "BATCH_SIZE",
    "SEED",
]:
    assert name in globals(), f"Missing {name}. Run Cell 1 first."

SAVE_DIR = "/content/drive/MyDrive/..."
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------- Locate the FedAvg checkpoint ----------------
fedavg_candidates = glob.glob(f"{SAVE_DIR}/*MNIST_FedAvg_02*.h5")
print("FedAvg candidates:", fedavg_candidates)

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(
        f"No FedAvg checkpoint found in {SAVE_DIR}. "
        f"Current contents: {os.listdir(SAVE_DIR)}"
    )

FEDAVG_PATH = fedavg_candidates[0]
print("Using FedAvg initialization:", FEDAVG_PATH)

DITTO_GLOBAL_PATH = f"{SAVE_DIR}/Ditto_Global.h5"
DITTO_PERSONAL_DIR = f"{SAVE_DIR}/Ditto_Personal_Models"
DITTO_LOGS_PATH = f"{SAVE_DIR}/Ditto_logs.json"
DITTO_METRICS_CSV = f"{SAVE_DIR}/Ditto_metrics_rows.csv"
os.makedirs(DITTO_PERSONAL_DIR, exist_ok=True)

# ---------------- Training configuration ----------------
ROUNDS = 10
LOCAL_STEPS_LIMIT = None
LOCAL_EPOCHS_GLOBAL = 1
LOCAL_EPOCHS_PERSONAL = 1

LR_GLOBAL = 1e-3
LR_PERSONAL = 1e-3
LAMBDA = 1e-2

# Early stopping is based on round-level personalized training accuracy
EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------- Rebuild the normalized MNIST test arrays ----------------
# This follows the same preprocessing used in Cell 1 so that
# specialization test sets are constructed consistently.
def load_normalized_mnist_test_like_cell1():
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_tr = y_tr.reshape(-1).astype("int32")
    y_te = y_te.reshape(-1).astype("int32")

    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7
    x_te = (x_te - mean) / std

    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_test_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(list(set(client_present[client_id]))), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


# ---------------- Helper functions ----------------
def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def eval_logits(model, ds):
    """
    Evaluate a model that returns logits.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


# ---------------- Local Ditto training ----------------
def local_train_global_ce(model, ds, lr=1e-3, epochs=1, steps_limit=None):
    """
    Train a local copy of the global model using standard cross-entropy loss.
    """
    optimizer = tf.keras.optimizers.Adam(lr)
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for _ in range(epochs):
        step_count = 0
        for x, y in ds:
            with tf.GradientTape() as tape:
                logits = model(x, training=True)
                loss = loss_fn(y, logits)

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

            bs = int(tf.shape(x)[0].numpy())
            total_loss += float(loss.numpy()) * bs
            total_n += bs
            acc.update_state(y, tf.nn.softmax(logits))

            step_count += 1
            if steps_limit is not None and step_count >= steps_limit:
                break

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_personal_ditto(personal_model, global_model, ds,
                               lr=1e-3, epochs=1, lam=1e-2, steps_limit=None):
    """
    Train a personalized model with the Ditto objective:

        CE(y, v_k(x)) + (lambda / 2) * ||v_k - w||^2

    The proximal term is computed using personal_model.weights and
    global_model.weights so that all weights remain aligned correctly,
    including BatchNorm parameters.
    """
    optimizer = tf.keras.optimizers.Adam(lr)
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for _ in range(epochs):
        step_count = 0
        for x, y in ds:
            with tf.GradientTape() as tape:
                logits = personal_model(x, training=True)
                ce = loss_fn(y, logits)

                prox = 0.0
                for personal_w, global_w in zip(personal_model.weights, global_model.weights):
                    prox += tf.reduce_sum(tf.square(personal_w - tf.stop_gradient(global_w)))
                prox = 0.5 * lam * prox

                loss = ce + prox

            grads = tape.gradient(loss, personal_model.trainable_variables)
            optimizer.apply_gradients(zip(grads, personal_model.trainable_variables))

            bs = int(tf.shape(x)[0].numpy())
            total_loss += float(loss.numpy()) * bs
            total_n += bs
            acc.update_state(y, tf.nn.softmax(logits))

            step_count += 1
            if steps_limit is not None and step_count >= steps_limit:
                break

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


# ---------------- Early stopping ----------------
class TrainAccEarlyStopping:
    """
    Stop training when the monitored personalized training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the initial FedAvg model ----------------
server = keras.models.load_model(FEDAVG_PATH)
dummy = tf.zeros([1] + list(server.input_shape[1:]), dtype=tf.float32)
_ = server(dummy, training=False)

# Initialize one personalized model per client from the same starting point
personal_models = [clone_model_with_weights(server) for _ in range(NUM_CLIENTS)]

base_loss, base_acc = eval_logits(server, test_ds)
print(f"\nInitial FedAvg model on full test set: loss={base_loss:.4f}, acc={base_acc:.4f}")

# Metric storage for JSON and CSV export
logs = {
    "round": [],
    "train_loss_global": [],
    "train_acc_global": [],
    "val_loss_global": [],
    "val_acc_global": [],
    "train_loss_personal": [],
    "train_acc_personal": [],
    "val_loss_personal": [],
    "val_acc_personal": [],
}

# ---------------- Federated Ditto training loop ----------------
for r in range(ROUNDS):
    client_ws = []

    trLg, trAg, vaLg, vaAg = [], [], [], []
    trLp, trAp, vaLp, vaAp = [], [], [], []

    for client_id in range(NUM_CLIENTS):
        # Global update: train a local copy of the current server model
        local_global = clone_model_with_weights(server)
        train_loss_global, train_acc_global = local_train_global_ce(
            local_global,
            client_train_datasets[client_id],
            lr=LR_GLOBAL,
            epochs=LOCAL_EPOCHS_GLOBAL,
            steps_limit=LOCAL_STEPS_LIMIT
        )
        val_loss_global, val_acc_global = eval_logits(local_global, client_val_datasets[client_id])

        trLg.append(train_loss_global)
        trAg.append(train_acc_global)
        vaLg.append(val_loss_global)
        vaAg.append(val_acc_global)
        client_ws.append(local_global.get_weights())

        # Personalized update: optimize the local model with the Ditto objective
        train_loss_personal, train_acc_personal = local_train_personal_ditto(
            personal_models[client_id],
            global_model=server,
            ds=client_train_datasets[client_id],
            lr=LR_PERSONAL,
            epochs=LOCAL_EPOCHS_PERSONAL,
            lam=LAMBDA,
            steps_limit=LOCAL_STEPS_LIMIT
        )
        val_loss_personal, val_acc_personal = eval_logits(
            personal_models[client_id],
            client_val_datasets[client_id]
        )

        trLp.append(train_loss_personal)
        trAp.append(train_acc_personal)
        vaLp.append(val_loss_personal)
        vaAp.append(val_acc_personal)

    # Aggregate the local global models to update the server
    server.set_weights(average_weights(client_ws))

    round_tr_lg = float(np.mean(trLg))
    round_tr_ag = float(np.mean(trAg))
    round_va_lg = float(np.mean(vaLg))
    round_va_ag = float(np.mean(vaAg))
    round_tr_lp = float(np.mean(trLp))
    round_tr_ap = float(np.mean(trAp))
    round_va_lp = float(np.mean(vaLp))
    round_va_ap = float(np.mean(vaAp))

    logs["round"].append(r)
    logs["train_loss_global"].append(round_tr_lg)
    logs["train_acc_global"].append(round_tr_ag)
    logs["val_loss_global"].append(round_va_lg)
    logs["val_acc_global"].append(round_va_ag)
    logs["train_loss_personal"].append(round_tr_lp)
    logs["train_acc_personal"].append(round_tr_ap)
    logs["val_loss_personal"].append(round_va_lp)
    logs["val_acc_personal"].append(round_va_ap)

    print(
        f"[Ditto][Round {r:02d}] "
        f"G(train) loss={round_tr_lg:.3f} acc={round_tr_ag:.3f} | "
        f"G(val) loss={round_va_lg:.3f} acc={round_va_ag:.3f} || "
        f"P(train) loss={round_tr_lp:.3f} acc={round_tr_ap:.3f} | "
        f"P(val) loss={round_va_lp:.3f} acc={round_va_ap:.3f}"
    )

    if stopper.step(round_tr_ap):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in personalized training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
global_test_loss, global_test_acc = eval_logits(server, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)
    spec_loss, spec_acc = eval_logits(personal_models[client_id], spec_ds_k)
    glob_loss, glob_acc = eval_logits(personal_models[client_id], test_ds)

    spec_losses.append(spec_loss)
    spec_accs.append(spec_acc)
    glob_losses.append(glob_loss)
    glob_accs.append(glob_acc)

print("\nFinal results")
print(f"Global server on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save models and logs ----------------
server.save(DITTO_GLOBAL_PATH)
print("\nSaved Ditto global model to:", DITTO_GLOBAL_PATH)

for client_id in range(NUM_CLIENTS):
    personal_models[client_id].save(f"{DITTO_PERSONAL_DIR}/ditto_personal_{client_id}.h5")
print("Saved Ditto personal models to:", DITTO_PERSONAL_DIR)

with open(DITTO_LOGS_PATH, "w") as f:
    json.dump(logs, f, indent=2)
print("Saved Ditto logs to:", DITTO_LOGS_PATH)

rows = [
    ["round"] + logs["round"],
    ["train_loss_global"] + logs["train_loss_global"],
    ["train_acc_global"] + logs["train_acc_global"],
    ["val_loss_global"] + logs["val_loss_global"],
    ["val_acc_global"] + logs["val_acc_global"],
    ["train_loss_personal"] + logs["train_loss_personal"],
    ["train_acc_personal"] + logs["train_acc_personal"],
    ["val_loss_personal"] + logs["val_loss_personal"],
    ["val_acc_personal"] + logs["val_acc_personal"],
]

with open(DITTO_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Saved Ditto metrics CSV to:", DITTO_METRICS_CSV)